In [ ]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Connect your Google account
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create Google Sheet (run only once)
sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

ws.append_row([
    "name", "item", "qty", "date", "eggless", "total"
])

print("Sheet ready:", sh.url)

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"]
    qty: int = Field(ge=1)
    date: str
    eggless: bool

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item."""
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else "Invalid day."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet."""
    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )
    except Exception as e:
        return f"Could not save the order: {e}"

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return f"Saved: {order.qty} x {order.item} for {order.name}, Rs {total}."

print("Day 4 Cell 2 completed!")

In [ ]:
from langchain.agents import create_agent

SYSTEM = """ROLE: You are the friendly order assistant for Sweet Crumbs Home Bakery.

FACTS:
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Always use get_price for prices and is_open for timings.
Never guess prices or timings.
Before log_order, make sure you have:
name, item, quantity and pickup date.
Ask the customer if any information is missing.
After saving, read the order back with the total.

FORMAT:
Under 60 words.
Be friendly.
Reply in English or Telugu, matching the customer."""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []

def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]
        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again."

print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))

print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))

In [ ]:
for m in history:
    m.pretty_print()

In [ ]:
# Convert customer messages into structured Order records

order_llm = llm.with_structured_output(Order)

messages = [
    "I'm Ravi, need 2 chocolate cakes, eggless, pickup on 24 Dec.",
    "My name is Priya. Please give me 3 cupcakes on 25 Dec, not eggless.",
    "This is Rahul. I want one plum cake, eggless, on 26 Dec.",
    "I'm Anu, order 4 cupcakes, eggless please, pickup 27 Dec.",
    "Name: Kiran. Need 2 plum cakes, regular, pickup on 28 Dec."
]

for msg in messages:
    print("Customer:", msg)
    try:
        result = order_llm.invoke(msg)
        print("Structured order:", result)
    except Exception as e:
        print("Error:", e)
    print("-" * 50)

In [ ]:
# Create a separate sheet for Activity 3

sh_activity = gc.create("Sweet Crumbs Activity 3")
ws_activity = sh_activity.sheet1

ws_activity.append_row([
    "name",
    "item",
    "quantity",
    "pickup_date",
    "eggless",
    "total"
])

print("Activity 3 Sheet created!")
print(sh_activity.url)

In [ ]:
from langchain.tools import tool

@tool
def log_sweet_crumbs_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """Save a customer's confirmed Sweet Crumbs cake order to the Activity 3 Google Sheet."""

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

        total = MENU[order.item] * order.qty

        ws_activity.append_row([
            order.name,
            order.item,
            order.qty,
            order.date,
            "yes" if order.eggless else "no",
            total
        ])

        return f"Order saved successfully. Total: Rs {total}."

    except Exception as e:
        return f"Order could not be saved: {e}"

print("Activity 3 tool created!")

In [ ]:
result = log_sweet_crumbs_order.invoke({
    "name": "Ravi",
    "item": "chocolate cake",
    "qty": 2,
    "date": "24 Dec",
    "eggless": True
})

print(result)

In [ ]:
from langchain.tools import tool

@tool("tool1")
def tool1(item: str, qty: int = 1) -> str:
    """does stuff"""
    price = MENU.get(item.lower().strip())

    if price is None:
        return "Item not available."

    return f"Total price: Rs {price * qty}"

print("Tool renamed to tool1")

In [ ]:
test_agent = create_agent(
    model=llm,
    tools=[tool1],
    system_prompt="You are a bakery assistant. Use the available tool to calculate prices."
)

result = test_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the price of 2 chocolate cakes?"}
    ]
})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

print("Original tools restored!")

In [ ]:
history = []

response = chat("How much are 3 cupcakes?")

print("Agent response:", response)

tool_calls = 0
model_messages = 0

for m in history:
    if getattr(m, "tool_calls", None):
        tool_calls += len(m.tool_calls)

    if m.type == "ai":
        model_messages += 1

print("Tool calls:", tool_calls)
print("AI messages:", model_messages)

for m in history:
    m.pretty_print()

In [ ]:
history = []

response = chat("How much are 3 cupcakes?")

print("Agent response:", response)

tool_calls = 0
model_messages = 0

for m in history:
    if getattr(m, "tool_calls", None):
        tool_calls += len(m.tool_calls)

    if m.type == "ai":
        model_messages += 1

print("\nTool calls:", tool_calls)
print("AI messages:", model_messages)

print("\nAgent Trace:")
for m in history:
    m.pretty_print()

In [ ]:
from google.colab import auth
auth.authenticate_user()